# Stage 5: retrieval evaluation

Run after training. Upload the current `src/` folder and CIFAR-100 files to the Colab runtime. The notebook extracts test embeddings, computes exact retrieval metrics, evaluates random and untrained baselines, runs the optional Faiss check, and saves the qualitative grid under `/content/results/`. The held-out test split should be evaluated only for a finalized configuration.

In [ ]:
from pathlib import Path
import sys
import torch

candidates = [Path.cwd(), Path('/content')]
candidates += [p for p in Path('/content').iterdir() if p.is_dir()]
ROOT = next((p for p in candidates if (p / 'src' / 'eval.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Upload the current src/ folder to /content or the notebook working folder.')
sys.path.insert(0, str(ROOT))

DATA_ROOT = next((p for p in (ROOT / 'data', Path('/content/data'))
                  if (p / 'cifar-100-python' / 'train').is_file()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Extract cifar-100-python under the project data/ folder or /content/data/.')

OUTPUT_ROOT = Path('/content')
CHECKPOINT = OUTPUT_ROOT / 'runs' / 'baseline' / 'best.pt'
if not CHECKPOINT.is_file():
    CHECKPOINT = ROOT / 'runs' / 'baseline' / 'best.pt'
if not CHECKPOINT.is_file():
    raise FileNotFoundError('Could not find runs/baseline/best.pt. Run Stage 4 first and keep its checkpoint in /content.')
print('Project:', ROOT)
print('Dataset:', DATA_ROOT)
print('Checkpoint:', CHECKPOINT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os
import subprocess

command = [
    sys.executable, '-m', 'src.eval', '--checkpoint', str(CHECKPOINT),
    '--data-root', str(DATA_ROOT), '--split-dir', str(OUTPUT_ROOT / 'splits'),
    '--output-dir', str(OUTPUT_ROOT / 'results'), '--device', 'auto',
    '--seed', '42', '--num-workers', '2', '--no-download',
]
print(' '.join(command))
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
subprocess.run(command, cwd=ROOT, env=env, check=True)

In [ ]:
import json
from IPython.display import Image, display

metrics_path = OUTPUT_ROOT / 'results' / 'test_metrics.json'
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))
display(Image(filename=str(OUTPUT_ROOT / 'results' / 'retrieval_grid.png')))

## Optional README table

The generator reads the JSON so README results are never typed by hand. Set `UPDATE_README = True` below if you want to update the README copy present in this runtime. The Faiss check is reported as unavailable when `faiss-cpu` is not installed; install it in the runtime if you want that comparison.

In [ ]:
UPDATE_README = False
if UPDATE_README:
    subprocess.run([
        sys.executable, str(ROOT / 'scripts' / 'update_readme_metrics.py'),
        '--metrics', str(OUTPUT_ROOT / 'results' / 'test_metrics.json'),
        '--readme', str(ROOT / 'README.md'),
    ], cwd=ROOT, check=True)
    print('README retrieval table updated.')